# Aplicação: Classificação de Prioridade de Consumo de Alimentos


### Integrantes do grupo:
#### Eduardo Santos Urbano
#### Giovanni França da Silva

---

O projeto Hisopi tem como objetivo auxiliar no gerenciamento de alimentos e na redução do desperdício, utilizando o conceito FEFO (*First Expired, First Out*), que prioriza o consumo dos alimentos que estão mais próximos do vencimento.

Nesta adaptação do exemplo, utilizaremos **2 qubits** para representar duas características de um alimento:

* **Qubit 0:** quantidade de dias restantes até o vencimento;
* **Qubit 1:** quantidade de alimento disponível.

Os valores são normalizados e utilizados como parâmetros das portas de rotação do circuito quântico.

O circuito utiliza uma porta **CNOT** entre os dois qubits para criar emaranhamento. Dessa forma, as duas características do alimento são consideradas conjuntamente pelo circuito.

O classificador possui duas possíveis classes:

* **+1:** alimento com alta prioridade de consumo;
* **-1:** alimento com baixa prioridade de consumo.

A classificação pode auxiliar o Hisopi na identificação de alimentos que devem receber maior atenção no gerenciamento do estoque, contribuindo para a aplicação do conceito FEFO e, consequentemente, para a redução do desperdício de alimentos.

In [1]:
!pip install pennylane matplotlib
!pip install pennylane numpy

import pennylane as qml
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 42.8 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1



# 1. Dados dos alimentos

## Cada alimento possui:
### dias até o vencimento e quantidade disponível

### Os valores estão normalizados entre 0 e pi.

In [2]:
X_dados = np.array([
    [0.10, 0.80],  # alta prioridade
    [0.20, 0.70],  # alta prioridade
    [0.30, 0.90],  # alta prioridade
    [0.80, 0.30],  # baixa prioridade
    [0.90, 0.20],  # baixa prioridade
    [0.70, 0.10]   # baixa prioridade
])

# +1 = alta prioridade
# -1 = baixa prioridade
Y_rotulos = np.array([
    1, 1, 1,
    -1, -1, -1
])

# 2. Dispositivo quântico com 2 Qubits

In [3]:
dev = qml.device("default.qubit", wires=2)

# 3. Classificador Quântico Variacional


In [4]:
@qml.qnode(dev)
def vqc_circuito(pesos_treinaveis, x_features):

    # A) Codificação dos dados
    # Qubit 0 = dias até o vencimento
    # Qubit 1 = quantidade disponível

    qml.RX(x_features[0], wires=0)
    qml.RY(x_features[1], wires=1)

    # B) Camada variacional
    qml.Rot(*pesos_treinaveis[0], wires=0)
    qml.Rot(*pesos_treinaveis[1], wires=1)

    # C) Emaranhamento
    qml.CNOT(wires=[0, 1])

    # D) Segunda camada de rotação
    qml.Rot(*pesos_treinaveis[2], wires=0)
    qml.Rot(*pesos_treinaveis[3], wires=1)

    # Medição do segundo qubit
    return qml.expval(qml.PauliZ(1))

# 4. Função de custo

In [5]:
def custo_classificador(pesos, X, Y):

    perdas = [
        (vqc_circuito(pesos, x) - y) ** 2
        for x, y in zip(X, Y)
    ]

    return qml.math.mean(qml.math.stack(perdas))

# 5. Inicialização dos pesos


In [6]:
np.random.seed(42)

pesos_iniciais = qml.numpy.random.random(
    (4, 3),
    requires_grad=True
)

opt = qml.GradientDescentOptimizer(
    stepsize=0.4
)

# 6. Treinamento


In [7]:
pesos_otimizados = pesos_iniciais

print("=== CLASSIFICADOR DE PRIORIDADE ===")

for epoca in range(20):

    pesos_otimizados, custo_val = opt.step_and_cost(
        lambda p: custo_classificador(
            p,
            X_dados,
            Y_rotulos
        ),
        pesos_otimizados
    )

    if epoca % 4 == 0 or epoca == 19:

        print(
            f"Época {epoca:2d} | "
            f"Custo: {custo_val:.4f}"
        )
# Exemplo: alimento com poucos dias até o vencimento e muito estoque

novo_alimento = np.array(
    [0.15, 0.75]
)

resultado = vqc_circuito(
    pesos_otimizados,
    novo_alimento
)

classe_prevista = (
    "ALTA PRIORIDADE"
    if resultado > 0
    else "BAIXA PRIORIDADE"
)

print("\n=== RESULTADO ===")

print(
    f"Saída do circuito: {resultado:.4f}"
)

print(
    f"Classificação: {classe_prevista}"
)

=== CLASSIFICADOR DE PRIORIDADE ===
Época  0 | Custo: 0.9403
Época  4 | Custo: 0.5887
Época  8 | Custo: 0.4863
Época 12 | Custo: 0.4148
Época 16 | Custo: 0.3793
Época 19 | Custo: 0.3676

=== RESULTADO ===
Saída do circuito: 0.3938
Classificação: ALTA PRIORIDADE


# 7. Testando um novo alimento


In [15]:
novo_alimento = np.array(
    [0.15, 0.75]
)

resultado = vqc_circuito(
    pesos_otimizados,
    novo_alimento
)

classe_prevista = (
    "ALTA PRIORIDADE"
    if resultado > 0
    else "BAIXA PRIORIDADE"
)

print("\n=== RESULTADO ===")

print(
    f"Saída do circuito: {resultado:.4f}"
)

print(
    f"Classificação: {classe_prevista}"
)


=== RESULTADO ===
Saída do circuito: -0.1117
Classificação: BAIXA PRIORIDADE
